# Lesson 13 Lab — NF4 and QLoRA: A 4-Bit Fine-Tuning Memory Ledger

**Puzzle:** If the frozen base model is four-bit, where does fine-tuning memory still go?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

QLoRA freezes a four-bit base, computes through a wider dtype, and trains LoRA matrices. The memory ledger still includes adapters, gradients, optimizer states, activations, temporary dequantization, and allocator reserve.

### Core mechanism

A rank-`r` adapter adds `ΔW = A·B` with roughly `r(in+out)` trainable parameters instead of `in×out`. NF4 provides a non-uniform 16-value codebook suited to normally distributed pretrained weights; double quantization compresses scale metadata.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "13-nf4-qlora"
device = require_cuda()
torch.manual_seed(2026 + 13)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Lower base storage enables larger models, but sequence length and activation checkpointing often dominate training memory. Adapter rank trades capacity against trainable state and compute.

### What this code tests

The lab combines a 7B-class arithmetic ledger with a real CUDA backward pass where only low-rank adapter tensors receive gradients.

**Experiment:** Build a 7B-class memory ledger and run a CUDA low-rank adapter forward/backward over a frozen fake-quantized base matrix.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
params=7_000_000_000; rank=16; hidden=4096; layers=32
ledger={"bf16_base_gib":round(params*2/2**30,3),"int4_base_ideal_gib":round(params*0.5/2**30,3),
        "lora_trainable_mib":round(layers*2*hidden*rank*2/2**20,3),"adam_states_mib":round(layers*2*hidden*rank*8/2**20,3)}
dim=1024; base=torch.randn(dim,dim,device=device); _,_,base_q=symmetric_quantize(base,bits=4,group_size=128); base_q=base_q.detach()
a=torch.nn.Parameter(torch.randn(dim,rank,device=device)*0.01); b=torch.nn.Parameter(torch.zeros(rank,dim,device=device))
x=torch.randn(64,dim,device=device); target=torch.randn(64,dim,device=device)
loss=torch.nn.functional.mse_loss(x@base_q.t()+(x@a)@b,target); loss.backward()
result=base_result(13,"pytorch-gpu"); result.update({"seven_b_ledger":ledger,"toy_loss":round(loss.item(),7),
    "base_requires_grad":base_q.requires_grad,"adapter_grad_finite":bool(torch.isfinite(a.grad).all() and torch.isfinite(b.grad).all()),
    "conclusion":"The frozen four-bit base reduced weight storage, while adapters, optimizer state, and activations remained separate costs."})


## 3. Inspect the evidence

Separate frozen base storage, trainable parameters, gradients, optimizer estimate, and activations.

### Acceptance and rollback gate

Reconcile theoretical and measured peak memory, confirm the base has no gradients, list compute dtype and optimizer, and validate downstream quality against a frozen baseline.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "adapter_grad_finite": true,
  "base_requires_grad": false,
  "conclusion": "The frozen four-bit base reduced weight storage, while adapters, optimizer state, and activations remained separate costs.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:40+00:00",
  "lesson": 13,
  "schema_version": 1,
  "seven_b_ledger": {
    "adam_states_mib": 32.0,
    "bf16_base_gib": 13.039,
    "int4_base_ideal_gib": 3.26,
    "lora_trainable_mib": 8.0
  },
  "toy_loss": 1036.6418457
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Four-bit base weights reduce one ledger line; sequence activations and adapter training state still control feasibility.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).